# Direction + magnitude hybrid models for USD/ZAR

This notebook predicts the 28-day USD/ZAR movement:

$$y_t = USDZAR_{t+28} - USDZAR_t$$

A positive value means **USD/ZAR rises (the rand weakens)**; a negative value means the rand strengthens. The LightGBM classifier estimates the probability of a positive move. A neural network separately estimates the absolute size of the move.

The combined signed forecast is:

$$\hat y = (2P(y>0)-1)\,\widehat{|y|}$$

This uses the classifier's certainty rather than applying a hard direction threshold. Evaluation uses expanding-window time-series cross-validation with a 28-row gap to reduce leakage from overlapping forecast horizons.

In [1]:
import random
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, root_mean_squared_error
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
print("PyTorch:", torch.__version__, "| device: CPU")

PyTorch: 2.13.0+cpu | device: CPU


## Load and engineer the model data

In [2]:
combined_data = (
    pd.read_parquet("../data/train.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)
combined_data["usd_zar_28_movement"] = (
    combined_data["usd_zar_28"] - combined_data["usd_zar"]
)
combined_data["interest_rate_diff"] = (
    combined_data["sa_repo_rate"] - combined_data["us_fed_funds"]
)
combined_data["sa_us_5y_yield_spread"] = (
    combined_data["sa_5y_yield"] - combined_data["us_5y_yield"]
)

# Equal-weight commodity basket. Dividing each price by its first observed value
# puts gold, platinum, coal, and iron ore on the same unit-free scale.
local_commodities = [
    "gold_usd_per_oz",
    "platinum_usd_per_oz",
    "richards_bay_coal_usd",
    "iron_ore_usd_per_tonne",
]
normalized_commodities = combined_data[local_commodities].div(
    combined_data[local_commodities].iloc[0]
)
combined_data["local_commodity_basket"] = normalized_commodities.mean(axis=1)

candidate_features = [
    "usd_zar",
    "local_commodity_basket",
    "brent_usd_per_barrel",
    "interest_rate_diff",
    "sa_us_5y_yield_spread",
    "sa_real_gdp",
    "sa_cpi",
    "sa_yoy_inflation",
    "sa_5y_cds_bp",
    "vix",
    "broad_usd_index",
    "usd_zar_1w_return",
    "usd_zar_1m_return",
    "usd_zar_3m_return",
    "usd_zar_1m_volatility",
]

X = combined_data[candidate_features].astype("float32")
y = combined_data["usd_zar_28_movement"].astype("float32")
y_direction = (y > 0).astype("int8")
y_magnitude = y.abs()

assert not X.isna().any().any()
print(f"{len(X):,} observations, {X.shape[1]} candidate predictors")
print(f"Positive-move share: {y_direction.mean():.3f}")
print("Commodity basket components:", ", ".join(local_commodities))
combined_data[
    ["date", *local_commodities, "local_commodity_basket", "usd_zar_28_movement"]
].head()

3,190 observations, 15 candidate predictors
Positive-move share: 0.504
Commodity basket components: gold_usd_per_oz, platinum_usd_per_oz, richards_bay_coal_usd, iron_ore_usd_per_tonne


        date  gold_usd_per_oz  ...  local_commodity_basket  usd_zar_28_movement
0 2008-10-10       855.400024  ...                1.000000               0.6658
1 2008-10-13       838.900024  ...                0.997386               0.8223
2 2008-10-14       836.299988  ...                1.012057               1.1054
3 2008-10-15       835.500000  ...                0.994110               1.0584
4 2008-10-16       801.500000  ...                0.950896               0.1641

[5 rows x 7 columns]

## Nested, time-aware feature selection and evaluation

Feature selection happens **inside each outer training fold**, so an outer test fold never influences its selected predictors. Within that training fold:

1. Keep the last 20% as an inner validation window, separated by a 28-row gap.
2. Fit one LightGBM classifier for direction and one LightGBM regressor for magnitude on the earlier observations.
3. Measure out-of-sample permutation importance on the inner validation window.
4. Average the normalized direction and magnitude importances, retain positive-importance predictors, and cap the set at eight (with a minimum of five).

The selected set is then used by both the direction classifier and neural magnitude model for the untouched outer fold. This nested design avoids selecting features on the same observations used to report RMSE.

In [3]:
N_SPLITS = 5
FORECAST_GAP = 28
MIN_FEATURES = 5
MAX_FEATURES = 8


def make_direction_model(seed):
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=350,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=30,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=seed,
        verbosity=-1,
        n_jobs=4,
    )


def make_selection_magnitude_model(seed):
    return lgb.LGBMRegressor(
        objective="regression",
        n_estimators=350,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=30,
        reg_lambda=1.0,
        random_state=seed,
        verbosity=-1,
        n_jobs=4,
    )


def _positive_normalize(values):
    values = np.maximum(np.asarray(values, dtype=float), 0.0)
    return values / values.sum() if values.sum() else values


def select_features_nested(X_train, y_train, seed):
    # Chronological inner holdout with a forecast-horizon gap.
    validation_size = max(150, int(0.20 * len(X_train)))
    validation_start = len(X_train) - validation_size
    inner_train_end = validation_start - FORECAST_GAP
    if inner_train_end <= 0:
        raise ValueError("Not enough training observations for nested selection")

    X_inner_train = X_train.iloc[:inner_train_end]
    X_inner_valid = X_train.iloc[validation_start:]
    y_inner_train = y_train.iloc[:inner_train_end]
    y_inner_valid = y_train.iloc[validation_start:]

    classifier = make_direction_model(seed)
    classifier.fit(X_inner_train, (y_inner_train > 0).astype("int8"))
    direction_pi = permutation_importance(
        classifier,
        X_inner_valid,
        (y_inner_valid > 0).astype("int8"),
        scoring="neg_log_loss",
        n_repeats=5,
        random_state=seed,
        n_jobs=1,
    ).importances_mean

    magnitude_selector = make_selection_magnitude_model(seed)
    magnitude_selector.fit(X_inner_train, y_inner_train.abs())
    magnitude_pi = permutation_importance(
        magnitude_selector,
        X_inner_valid,
        y_inner_valid.abs(),
        scoring="neg_root_mean_squared_error",
        n_repeats=5,
        random_state=seed,
        n_jobs=1,
    ).importances_mean

    combined_importance = (
        _positive_normalize(direction_pi) + _positive_normalize(magnitude_pi)
    ) / 2.0
    ranking = pd.Series(combined_importance, index=X_train.columns).sort_values(
        ascending=False
    )
    n_positive = int((ranking > 0).sum())
    n_selected = min(MAX_FEATURES, max(MIN_FEATURES, n_positive))
    return ranking.head(n_selected).index.tolist(), ranking


def evaluate_hybrid(name, magnitude_factory):
    rows = []
    selection_rows = []
    splitter = TimeSeriesSplit(n_splits=N_SPLITS, gap=FORECAST_GAP)

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X), start=1):
        X_train_all, X_test_all = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        selected, importance = select_features_nested(
            X_train_all, y_train, SEED + fold
        )
        X_train = X_train_all[selected]
        X_test = X_test_all[selected]

        direction_model = make_direction_model(SEED + fold)
        direction_model.fit(X_train, y_direction.iloc[train_idx])
        p_up = direction_model.predict_proba(X_test)[:, 1]

        magnitude_model = magnitude_factory(SEED + fold)
        magnitude_model.fit(X_train, y_magnitude.iloc[train_idx])
        predicted_magnitude = np.maximum(0.0, magnitude_model.predict(X_test))
        hybrid_prediction = (2.0 * p_up - 1.0) * predicted_magnitude

        rows.append({
            "model": name,
            "fold": fold,
            "n_features": len(selected),
            "selected_features": ", ".join(selected),
            "direction_accuracy": accuracy_score(y_direction.iloc[test_idx], p_up >= 0.5),
            "mean_direction_certainty": np.maximum(p_up, 1.0 - p_up).mean(),
            "magnitude_rmse": root_mean_squared_error(
                y_magnitude.iloc[test_idx], predicted_magnitude
            ),
            "hybrid_rmse": root_mean_squared_error(y_test, hybrid_prediction),
            "zero_change_rmse": root_mean_squared_error(
                y_test, np.zeros(len(y_test))
            ),
        })
        for feature in selected:
            selection_rows.append({
                "model": name,
                "fold": fold,
                "feature": feature,
                "combined_permutation_importance": importance[feature],
            })
        print(
            f"{name} | fold {fold}: "
            f"{len(selected)} features, accuracy={rows[-1]['direction_accuracy']:.3f}, "
            f"hybrid RMSE={rows[-1]['hybrid_rmse']:.5f}"
        )
        print("  selected:", ", ".join(selected))

    return pd.DataFrame(rows), pd.DataFrame(selection_rows)

## 1. LightGBM direction certainty + multilayer perceptron magnitude

In [4]:
def make_mlp(seed):
    return Pipeline([
        ("scale", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(64, 32, 16),
            activation="relu",
            solver="adam",
            alpha=1e-3,
            batch_size=128,
            learning_rate_init=1e-3,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=25,
            random_state=seed,
        )),
    ])


mlp_results, mlp_selection = evaluate_hybrid("LightGBM + MLP", make_mlp)
mlp_results

LightGBM + MLP | fold 1: 8 features, accuracy=0.540, hybrid RMSE=0.42771
  selected: broad_usd_index, brent_usd_per_barrel, sa_5y_cds_bp, usd_zar_1w_return, sa_us_5y_yield_spread, usd_zar, usd_zar_1m_return, interest_rate_diff
LightGBM + MLP | fold 2: 8 features, accuracy=0.614, hybrid RMSE=0.31356
  selected: sa_us_5y_yield_spread, sa_5y_cds_bp, usd_zar_1m_return, usd_zar_1w_return, usd_zar_1m_volatility, usd_zar_3m_return, sa_yoy_inflation, vix
LightGBM + MLP | fold 3: 8 features, accuracy=0.467, hybrid RMSE=1.43058
  selected: usd_zar, sa_yoy_inflation, sa_5y_cds_bp, broad_usd_index, local_commodity_basket, usd_zar_3m_return, sa_us_5y_yield_spread, usd_zar_1m_volatility
LightGBM + MLP | fold 4: 8 features, accuracy=0.584, hybrid RMSE=0.58317
  selected: sa_5y_cds_bp, sa_us_5y_yield_spread, local_commodity_basket, usd_zar_1m_volatility, usd_zar_1m_return, sa_yoy_inflation, brent_usd_per_barrel, usd_zar_1w_return
LightGBM + MLP | fold 5: 8 features, accuracy=0.580, hybrid RMSE=0.94073

            model  fold  ...  hybrid_rmse zero_change_rmse
0  LightGBM + MLP     1  ...     0.427715         0.296887
1  LightGBM + MLP     2  ...     0.313560         0.315234
2  LightGBM + MLP     3  ...     1.430579         0.624738
3  LightGBM + MLP     4  ...     0.583166         0.567337
4  LightGBM + MLP     5  ...     0.940729         0.779810

[5 rows x 9 columns]

## 2. LightGBM direction certainty + FT-Transformer magnitude

The FT-Transformer tokenizes every numerical feature, adds a learned classification token, and uses self-attention to model feature interactions. It is structurally different from a conventional fully connected MLP.

In [5]:
class FTTransformerNet(nn.Module):
    def __init__(self, n_features, d_token=24, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.feature_weight = nn.Parameter(torch.empty(n_features, d_token))
        self.feature_bias = nn.Parameter(torch.empty(n_features, d_token))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.xavier_uniform_(self.feature_weight)
        nn.init.normal_(self.feature_bias, std=0.01)

        layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1),
            nn.Softplus(),
        )

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.feature_weight + self.feature_bias
        cls = self.cls_token.expand(x.size(0), -1, -1)
        encoded = self.transformer(torch.cat([cls, tokens], dim=1))
        return self.head(encoded[:, 0]).squeeze(-1)


class FTTransformerRegressor:
    def __init__(self, seed=42, epochs=45, batch_size=256, learning_rate=1e-3):
        self.seed = seed
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        self.scaler_ = StandardScaler()
        X_scaled = self.scaler_.fit_transform(X).astype("float32")
        y_array = np.asarray(y, dtype="float32")
        dataset = TensorDataset(torch.from_numpy(X_scaled), torch.from_numpy(y_array))
        generator = torch.Generator().manual_seed(self.seed)
        loader = DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=True,
            generator=generator,
        )

        self.model_ = FTTransformerNet(X_scaled.shape[1])
        optimizer = torch.optim.AdamW(
            self.model_.parameters(), lr=self.learning_rate, weight_decay=1e-4
        )
        loss_fn = nn.MSELoss()
        self.model_.train()
        for _ in range(self.epochs):
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = loss_fn(self.model_(xb), yb)
                loss.backward()
                optimizer.step()
        return self

    def predict(self, X):
        X_scaled = self.scaler_.transform(X).astype("float32")
        self.model_.eval()
        with torch.no_grad():
            return self.model_(torch.from_numpy(X_scaled)).numpy()


def make_ft_transformer(seed):
    return FTTransformerRegressor(seed=seed)


transformer_results, transformer_selection = evaluate_hybrid(
    "LightGBM + FT-Transformer", make_ft_transformer
)
transformer_results

LightGBM + FT-Transformer | fold 1: 8 features, accuracy=0.540, hybrid RMSE=0.32010
  selected: broad_usd_index, brent_usd_per_barrel, sa_5y_cds_bp, usd_zar_1w_return, sa_us_5y_yield_spread, usd_zar, usd_zar_1m_return, interest_rate_diff
LightGBM + FT-Transformer | fold 2: 8 features, accuracy=0.614, hybrid RMSE=0.29202
  selected: sa_us_5y_yield_spread, sa_5y_cds_bp, usd_zar_1m_return, usd_zar_1w_return, usd_zar_1m_volatility, usd_zar_3m_return, sa_yoy_inflation, vix
LightGBM + FT-Transformer | fold 3: 8 features, accuracy=0.467, hybrid RMSE=0.73457
  selected: usd_zar, sa_yoy_inflation, sa_5y_cds_bp, broad_usd_index, local_commodity_basket, usd_zar_3m_return, sa_us_5y_yield_spread, usd_zar_1m_volatility
LightGBM + FT-Transformer | fold 4: 8 features, accuracy=0.584, hybrid RMSE=0.54743
  selected: sa_5y_cds_bp, sa_us_5y_yield_spread, local_commodity_basket, usd_zar_1m_volatility, usd_zar_1m_return, sa_yoy_inflation, brent_usd_per_barrel, usd_zar_1w_return
LightGBM + FT-Transformer | 

                       model  fold  ...  hybrid_rmse zero_change_rmse
0  LightGBM + FT-Transformer     1  ...     0.320104         0.296887
1  LightGBM + FT-Transformer     2  ...     0.292016         0.315234
2  LightGBM + FT-Transformer     3  ...     0.734573         0.624738
3  LightGBM + FT-Transformer     4  ...     0.547429         0.567337
4  LightGBM + FT-Transformer     5  ...     0.887087         0.779810

[5 rows x 9 columns]

## Final out-of-sample comparison and feature-selection stability

Besides mean fold metrics, the selection frequency table shows how stable each predictor is across outer folds. Stable features are more trustworthy than a feature chosen in only one particular period.

In [6]:
all_results = pd.concat([mlp_results, transformer_results], ignore_index=True)
summary = (
    all_results.groupby("model")
    .agg(
        mean_features_selected=("n_features", "mean"),
        direction_accuracy=("direction_accuracy", "mean"),
        mean_direction_certainty=("mean_direction_certainty", "mean"),
        magnitude_rmse=("magnitude_rmse", "mean"),
        hybrid_rmse=("hybrid_rmse", "mean"),
        hybrid_rmse_std=("hybrid_rmse", "std"),
        zero_change_rmse=("zero_change_rmse", "mean"),
    )
    .sort_values("hybrid_rmse")
)
summary["rmse_improvement_vs_zero"] = (
    summary["zero_change_rmse"] - summary["hybrid_rmse"]
)
display(summary.round(5))

all_selection = pd.concat([mlp_selection, transformer_selection], ignore_index=True)
selection_stability = (
    all_selection.groupby(["model", "feature"])
    .agg(
        folds_selected=("fold", "nunique"),
        mean_importance=("combined_permutation_importance", "mean"),
    )
    .sort_values(["model", "folds_selected", "mean_importance"], ascending=[True, False, False])
)
selection_stability

                           mean_features_selected  ...  rmse_improvement_vs_zero
model                                              ...                          
LightGBM + FT-Transformer                     8.0  ...                  -0.03944
LightGBM + MLP                                8.0  ...                  -0.22235

[2 rows x 8 columns]
                                                  folds_selected  mean_importance
model                     feature                                                
LightGBM + FT-Transformer sa_us_5y_yield_spread                5         0.223796
                          sa_5y_cds_bp                         5         0.162306
                          sa_yoy_inflation                     4         0.102164
                          usd_zar_1m_volatility                4         0.088616
                          usd_zar_1w_return                    4         0.069513
                          broad_usd_index                      3         0.25427